# beam_center: real motor calibration notebook

This notebook is meant to be used **before** any automatic centering on real hardware.

Its goal is to calibrate the basic response of the system:

- motor movement direction versus image motion
- approximate pixel-per-step sensitivity
- safe step size
- settling time after a move

It does **not** run a closed-loop centering algorithm by itself.
Use the collected numbers to configure a later centering notebook safely.

## Safety notes

Use very small steps first.

Check the actual motor limits and interlocks in the control system before writing any value from this notebook.

If the beam can be lost or the image can saturate, stop and recover the operating point before continuing.

In [ ]:
import time
from p4p.client.thread import Context

ctx = Context("pva")

# Set these PVs for the real system you want to calibrate.
CAM = "LAB:REAL:CAM01"
MOTX = "LAB:REAL:HMIR"
MOTY = "LAB:REAL:VMIR"


def unwrap_scalar(value):
    if hasattr(value, "value"):
        value = value.value
    if isinstance(value, dict) and "value" in value:
        value = value["value"]
    return value


def get_scalar(pv, timeout=2.0):
    return unwrap_scalar(ctx.get(pv, timeout=timeout))


def put_scalar(pv, value, wait=True, timeout=5.0):
    ctx.put(pv, value, wait=wait, timeout=timeout)


def read_motor_state(motor_pv):
    return {
        "RBV": float(get_scalar(f"{motor_pv}.RBV")),
        "VAL": float(get_scalar(f"{motor_pv}.VAL")),
        "DMOV": float(get_scalar(f"{motor_pv}.DMOV")),
    }


def read_beam_centroid(camera_prefix):
    return {
        "x": float(get_scalar(f"{camera_prefix}:Stats1:CentroidX_RBV")),
        "y": float(get_scalar(f"{camera_prefix}:Stats1:CentroidY_RBV")),
    }


def wait_motor_done(motor_pv, timeout=20.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if float(get_scalar(f"{motor_pv}.DMOV")) == 1.0:
            return True
        time.sleep(0.1)
    return False


def move_motor_relative(motor_pv, delta, move_timeout=20.0):
    start = float(get_scalar(f"{motor_pv}.RBV"))
    put_scalar(f"{motor_pv}.VAL", start + delta, wait=False)
    ok = wait_motor_done(motor_pv, timeout=move_timeout)
    end = float(get_scalar(f"{motor_pv}.RBV"))
    return {
        "start": start,
        "target": start + delta,
        "end": end,
        "done": ok,
    }

In [ ]:
# Calibration parameters.
# Start conservatively and only increase the step after you verify the direction.
PARAMS = {
    "test_step_x": 0.01,
    "test_step_y": 0.01,
    "settle_time": 1.5,
    "move_timeout": 20.0,
    "restore_after_test": True,
}

PARAMS

In [ ]:
# Check the current operating point.
put_scalar(f"{CAM}:Stats1:ComputeCentroid", 1)
time.sleep(0.3)

{
    "motor_x": read_motor_state(MOTX),
    "motor_y": read_motor_state(MOTY),
    "beam": read_beam_centroid(CAM),
}

## X-axis direction and sensitivity check

This cell applies one small X step, waits, measures the centroid change, and optionally returns the motor to the starting point.

Interpretation:

- if `beam_dx` is positive after a positive motor step, X polarity is positive
- if `beam_dx` is negative after a positive motor step, X polarity is negative
- `beam_dx / motor_dx` gives a first estimate of pixels per motor unit

In [ ]:
before_beam = read_beam_centroid(CAM)
before_motor = read_motor_state(MOTX)
move = move_motor_relative(MOTX, PARAMS["test_step_x"], move_timeout=PARAMS["move_timeout"])
time.sleep(PARAMS["settle_time"])
after_beam = read_beam_centroid(CAM)

result_x = {
    "before_motor": before_motor,
    "move": move,
    "before_beam": before_beam,
    "after_beam": after_beam,
    "motor_dx": move["end"] - move["start"],
    "beam_dx": after_beam["x"] - before_beam["x"],
    "beam_dy": after_beam["y"] - before_beam["y"],
}

if PARAMS["restore_after_test"]:
    move_motor_relative(MOTX, -PARAMS["test_step_x"], move_timeout=PARAMS["move_timeout"])
    time.sleep(PARAMS["settle_time"])

result_x

## Y-axis direction and sensitivity check

This does the same test for the vertical motor.
Watch both centroid axes, because cross-coupling can be non-negligible on real systems.

In [ ]:
before_beam = read_beam_centroid(CAM)
before_motor = read_motor_state(MOTY)
move = move_motor_relative(MOTY, PARAMS["test_step_y"], move_timeout=PARAMS["move_timeout"])
time.sleep(PARAMS["settle_time"])
after_beam = read_beam_centroid(CAM)

result_y = {
    "before_motor": before_motor,
    "move": move,
    "before_beam": before_beam,
    "after_beam": after_beam,
    "motor_dy": move["end"] - move["start"],
    "beam_dx": after_beam["x"] - before_beam["x"],
    "beam_dy": after_beam["y"] - before_beam["y"],
}

if PARAMS["restore_after_test"]:
    move_motor_relative(MOTY, -PARAMS["test_step_y"], move_timeout=PARAMS["move_timeout"])
    time.sleep(PARAMS["settle_time"])

result_y

In [ ]:
def safe_pixels_per_unit(beam_delta, motor_delta):
    if abs(motor_delta) < 1e-12:
        return None
    return beam_delta / motor_delta

def suggested_sign(pixels_per_unit):
    if pixels_per_unit is None or pixels_per_unit == 0:
        return None
    return -1.0 if pixels_per_unit > 0 else 1.0

def suggested_gain(pixels_per_unit, response_fraction=0.2):
    if pixels_per_unit is None or pixels_per_unit == 0:
        return None
    return response_fraction / abs(pixels_per_unit)

ppu_x = safe_pixels_per_unit(result_x["beam_dx"], result_x["motor_dx"])
ppu_y = safe_pixels_per_unit(result_y["beam_dy"], result_y["motor_dy"])

POST_CALIBRATION = {
    "sign_x": suggested_sign(ppu_x),
    "sign_y": suggested_sign(ppu_y),
    "pixels_per_motor_unit_x": ppu_x,
    "pixels_per_motor_unit_y": ppu_y,
    "suggested_gain_x": suggested_gain(ppu_x),
    "suggested_gain_y": suggested_gain(ppu_y),
    "suggested_settle_time": PARAMS["settle_time"],
    "suggested_move_timeout": PARAMS["move_timeout"],
    "suggested_tolerance": 5.0,
    "suggested_max_step_x": abs(PARAMS["test_step_x"]),
    "suggested_max_step_y": abs(PARAMS["test_step_y"]),
    "cross_coupling_from_x_test": result_x["beam_dy"],
    "cross_coupling_from_y_test": result_y["beam_dx"],
}

POST_CALIBRATION

## Post-calibration parameter list

This section turns the measured X and Y test results into a clear parameter set for the real centering loop.

Review the generated values before using them on hardware. In particular:

- confirm `sign_x` and `sign_y` match the observed motion direction
- check that the suggested gains are conservative enough
- keep `max_step_x` and `max_step_y` small for the first centering runs

## Suggested next step

After these checks you should be able to estimate:

- `sign_x` and `sign_y` for the control law
- rough `pixels_per_motor_unit_x` and `pixels_per_motor_unit_y`
- a conservative gain such as `gain ~= 0.2 / pixels_per_motor_unit`
- a realistic settling time

Only after that should you run a real centering notebook.

## Real beam centering after calibration

Only run this section after the calibration cells above have given you a reasonable estimate for:

- direction sign for each motor
- usable gain magnitude
- settling time
- safe motion range

This centering loop is still intentionally conservative. It uses explicit signs and a maximum correction per iteration so that the calibration results are applied in a controlled way.

In [ ]:
# Fill these values using POST_CALIBRATION and your hardware judgement.
# The generated suggestions are a starting point, not an authorization to move aggressively.
CENTERING = {
    "max_iterations": 10,
    "gain_x": POST_CALIBRATION["suggested_gain_x"] or 0.001,
    "gain_y": POST_CALIBRATION["suggested_gain_y"] or 0.001,
    "sign_x": POST_CALIBRATION["sign_x"] or 1.0,
    "sign_y": POST_CALIBRATION["sign_y"] or 1.0,
    "tolerance": POST_CALIBRATION["suggested_tolerance"],
    "settle_time": POST_CALIBRATION["suggested_settle_time"],
    "move_timeout": POST_CALIBRATION["suggested_move_timeout"],
    "max_step_x": POST_CALIBRATION["suggested_max_step_x"],
    "max_step_y": POST_CALIBRATION["suggested_max_step_y"],
}

CENTERING

In [ ]:
def clamp(value, limit):
    return max(-abs(limit), min(abs(limit), value))

def read_overlay_center(camera_prefix):
    px = float(get_scalar(f"{camera_prefix}:Overlay1:1:PositionX_RBV"))
    py = float(get_scalar(f"{camera_prefix}:Overlay1:1:PositionY_RBV"))
    sx = float(get_scalar(f"{camera_prefix}:Overlay1:1:SizeX_RBV"))
    sy = float(get_scalar(f"{camera_prefix}:Overlay1:1:SizeY_RBV"))
    return {
        "x": px + sx / 2.0,
        "y": py + sy / 2.0,
    }

def run_real_beam_center(config):
    put_scalar(f"{CAM}:Stats1:ComputeCentroid", 1)
    time.sleep(0.3)
    history = []

    for iteration in range(config["max_iterations"]):
        beam = read_beam_centroid(CAM)
        target = read_overlay_center(CAM)
        err_x = target["x"] - beam["x"]
        err_y = target["y"] - beam["y"]

        step_x = clamp(err_x * config["gain_x"] * config["sign_x"], config["max_step_x"])
        step_y = clamp(err_y * config["gain_y"] * config["sign_y"], config["max_step_y"])

        row = {
            "iteration": iteration,
            "beam": beam,
            "target": target,
            "err_x": err_x,
            "err_y": err_y,
            "step_x": step_x,
            "step_y": step_y,
        }
        history.append(row)

        print(
            f"[{iteration}] beam=({beam['x']:.1f}, {beam['y']:.1f}) "
            f"target=({target['x']:.1f}, {target['y']:.1f}) "
            f"err=({err_x:.1f}, {err_y:.1f}) "
            f"step=({step_x:.4f}, {step_y:.4f})"
        )

        if abs(err_x) < config["tolerance"] and abs(err_y) < config["tolerance"]:
            print(f"Converged in {iteration + 1} iterations")
            break

        move_motor_relative(MOTX, step_x, move_timeout=config["move_timeout"])
        move_motor_relative(MOTY, step_y, move_timeout=config["move_timeout"])
        time.sleep(config["settle_time"])

        if not wait_motor_done(MOTX, timeout=config["move_timeout"]):
            print("X motor timeout")
            break
        if not wait_motor_done(MOTY, timeout=config["move_timeout"]):
            print("Y motor timeout")
            break

    return history

In [ ]:
# Final centering run after calibration.
# Review CENTERING first, especially sign_x, sign_y, max_step_x, and max_step_y.
centering_history = run_real_beam_center(CENTERING)
centering_history[-3:] if centering_history else []